In [19]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()

True

# 提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

## 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [31]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

from langchain.chat_models import init_chat_model
import os
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")
# 初始化模型
model = init_chat_model(
    model="qwen3.6-max-preview",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=base_url,
    api_key=api_key
)

# # 创建智能体
# agent = create_agent(
#     model = "deepseek-chat"
# )

agent= create_agent(model=model)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都在哪里?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


月球**没有首都**。

月球是地球的天然卫星，目前**没有任何国家、政府、城市或常住人口**，因此不存在“首都”或任何行政区划的概念。根据1967年联合国《外层空间条约》，月球及其他天体**不属于任何国家**，任何国家不得对其主张主权或进行领土划分。

虽然人类已通过阿波罗计划等多次登月，并且中美欧等国家和机构正在规划未来的月球科研站或常驻基地，但这些都将是**国际合作或科学探索性质的设施**，而非城市或行政中心。

如果您对月球探测历史、未来月球基地计划或相关国际法规感兴趣，我可以为您进一步介绍！ 🌕

In [21]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
# agent = create_agent(
#     model = "deepseek-chat",
#     system_prompt="你是一个科幻作家，根据用户的要求创建一个首都城市."
# )

agent= create_agent(model=model)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都在哪里?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


月球是地球的自然卫星，并不是国家或任何政治实体，因此**没有首都**。目前月球上也没有人类常驻居民、政府机构或城市。根据联合国《外层空间条约》，月球不属于任何国家，而是全人类共同探索与和平利用的领域。

# 2.Few-Shot examples

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习模式并生成更准确的响应。

In [33]:

system_prompt = """
你是一个科幻作家，根据用户的要求创建一个太空之都。

用户：月球的首都是什么？
科幻作家：月华城（Lunara）

用户：火星的首都是什么？
科幻作家：赤晶城（Rubrum）

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


科幻作家：云霆城（Aeropolis）

# 3.结构化输出
结构化输出允许你指定模型的输出格式，从而使模型的输出更易于解析和使用。



In [30]:

system_prompt = """
你是一个科幻作家，根据用户的要求创建一个太空之都。请遵守以下结构，不要加markdown样式。

Name：首都的名字

Location：它所在的地方

Vibe：2-3个词来描述它的氛围

Economy：主要产业

"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="月球的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Name：新月阁

Location：位于月球南极的沙克尔顿环形山边缘，环抱着永恒冰冕

Vibe：静谧、科技感

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


、孤绝

Economy：氦-3开采、量子计算机芯片制造、月球观光

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是设定好一个数据类型即可。


In [24]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [29]:
# 然后，我们就可以创建智能体并设置结构化输出的格式了。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [26]:
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='b7762413-acc6-4452-b4a6-08576eb1cfcc'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 330, 'total_tokens': 419, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 330}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_058df29938_prod0820_fp8_kvcache_20260402', 'id': 'f3e737eb-bc8c-4521-b5de-ef94adf8a8ea', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd004-5043-72d1-a6da-4add0d4f7f7d-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '月球首都', 'location': '月球', 'vibe': '未来科幻', 'economy': '高科技经济'}, 'id': 'call_00_4tRD0uLe0m6ZWVEyasQ9PAbm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 330, 'output_to

In [27]:
city = response['structured_response']
city

CapitalInfo(name='月球首都', location='月球', vibe='未来科幻', economy='高科技经济')

In [28]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

月球首都位于月球，是一座未来科幻的城市，其主要产业包括高科技经济。
